In [7]:
!pip install pdbparser biopython mygene openpyxl freesasa py3Dmol -q
import sys
!{sys.executable} -m pip install pymol-open-source -q

In [11]:
import pymol
from pymol import cmd
import matplotlib.colors as mcolors
import pandas as pd

# --- Parameters ---
pdb_file = 'Mouse STING homodimer(ranked_0).pdb'  # Ensure this file is in the same directory

excel_file = 'inferred_weights_Primary_vs_Met_4T1.csv'
residue_df = pd.read_csv(excel_file)

# Calculate mean and standard deviation for 'additive_fitness'
mean_fitness = residue_df['additive'].mean()
std_fitness = residue_df['additive'].std()

# Filter for residues where 'additive_fitness' is greater than 1 standard deviation above the mean
filtered_residue_df = residue_df[residue_df['additive'] > (std_fitness)].copy()

# Set the color for filtered residues to black
filtered_residue_df['color'] = '#000000' # Black color

residue_colors = dict(zip(filtered_residue_df['Unnamed: 0'].astype(str), filtered_residue_df['color']))
output_png = "mets_hi_monomer_dimer_3D.png"


# --- Initialize PyMOL ---
# '-cq' runs PyMOL in command-line quiet mode
pymol.finish_launching(['pymol', '-cq'])
cmd.reinitialize()

# Load structure and set base style
cmd.load(pdb_file, 'prot')
cmd.hide('everything', 'all')
cmd.show('cartoon', 'prot')
cmd.color('gray70', 'prot')

# Visual clean-up (No shadows, white background)
cmd.bg_color('white')
cmd.set('ray_shadows', 0)
cmd.set('depth_cue', 0)
cmd.set('ray_opaque_background', 0)

# --- Apply Sphere Coloring ---
for resi, hex_color in residue_colors.items():
    # Define a unique name for the color in PyMOL
    color_name = f'color_{resi}'
    rgb = mcolors.to_rgb(hex_color)
    cmd.set_color(color_name, rgb)

    # Select and show spheres
    selection = f'resi {resi}'
    cmd.show('spheres', selection)
    cmd.color(color_name, selection)

# Center the view on the highlighted spheres
cmd.zoom('spheres', buffer=5)

cmd.set_view([0.479985863,   -0.019433912,   -0.877059817,
    0.655858099,   -0.656033754,    0.373463154,
    -0.582640409,   -0.754482031,   -0.302139550,
    0.000000000,    0.000000000, -306.655059814,
    0.225017548,   -0.535675049,   -0.549560547,
  261.217132568,  352.092956543,  -20.000000000 ])

# --- Render and Save ---
cmd.ray(1200, 900)
cmd.png(output_png, dpi=300)
print(f'Successfully saved image to {output_png}')

Successfully saved image to mets_hi_monomer_dimer_3D.png
